In [1]:
from src import model_factory
tokenizer, model = model_factory("templates/model.yaml")

 >> WARNING from QuantBeyoundEfficiency:  >> HF_TOKEN is not set. Copy .env.example to .env and add your  >> Hugging Face token (https://huggingface.co/settings/tokens).
 >> Model Factory for templates/model.yaml


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [20]:
print(model)
def activation_hook(module, input,output): 
    print("Activations:", output)
    print(output.shape)
def aggregate_attention_matrix(module, input,output): 
    print("Activations:", output)
    print(output.shape)

hook_handle = model.model.layers[27].mlp.act_fn.register_forward_hook(activation_hook)
input = tokenizer("test", return_tensors="pt").to("cuda:0")
output = model.generate(**input)
#important to remove hook when done
hook_handle.remove()


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
import torch 

quant_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-1.7B",
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0",
        output_attentions= True,
    )
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
from transformer_lens.model_bridge import TransformerBridge
bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")
logits, cache = bridge.run_with_cache("Hello world")

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "c:\Users\Ramneek\anaconda3\envs\arena-env2\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.